# El invernadero: ATE, posterior y decisiones

**Fundamentos matemáticos de la causalidad** · Matias Moran

Este notebook acompaña la charla *Inferencia bayesiana: pensar con incertidumbre*. Usa una simulación reproducible del ejemplo del invernadero de *Statistical Rethinking* (cap. 6) para responder tres preguntas:

1. `P(ATE > 0 | datos)`
2. `P(0.5 < ATE < 1.5 cm | datos)`
3. `P(≥5 de 10 plantas nuevas tratadas > 14 cm | datos)`

El modelo principal es lineal en centímetros, para que `β_T` sea directamente comparable con el ATE de la materia. Los datos son simulados, no reales. La semilla es `1`.


## 0. Librerías y configuración

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pymc as pm
import arviz as az

RANDOM_SEED = 1

plt.rcParams["figure.figsize"] = (8, 4)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False
pd.set_option("display.precision", 3)


## 1. Simulación del invernadero

Generamos 100 plantas con el mismo mecanismo del libro:

- altura inicial `h0 ~ Normal(10, 2)`
- tratamiento `T` aleatorizado 50/50
- hongo `F ~ Bernoulli(0.5 - 0.4 T)`
- altura final `h1 = h0 + Normal(5 - 3 F, 1)`

En este DGP, el tratamiento reduce la probabilidad de hongo en 0.4 y el hongo reduce el crecimiento en 3 cm. Por eso el ATE verdadero es `3 × 0.4 = 1.2 cm`.


In [ ]:
rng = np.random.default_rng(RANDOM_SEED)

N = 100
h0 = rng.normal(10, 2, N)
treatment = np.repeat([0, 1], N // 2)
fungus = rng.binomial(1, 0.5 - 0.4 * treatment)
h1 = h0 + rng.normal(5 - 3 * fungus, 1.0, N)

df = pd.DataFrame({
    "h0": h0,
    "treatment": treatment,
    "fungus": fungus,
    "h1": h1,
})
df["h0_c"] = df["h0"] - df["h0"].mean()
df.head()


In [ ]:
df.groupby("treatment").agg(
    n=("h1", "size"),
    h0_mean=("h0", "mean"),
    h1_mean=("h1", "mean"),
    fungus_mean=("fungus", "mean"),
).round(2)


La aleatorización deja los grupos razonablemente balanceados en altura inicial. La diferencia de medias de `h1` es el estimador de ATE de la materia.


## 2. El ATE de la materia

In [ ]:
treated = df.loc[df["treatment"] == 1, "h1"]
control = df.loc[df["treatment"] == 0, "h1"]

ate_hat = treated.mean() - control.mean()
print(f"ATE_hat = {ate_hat:.3f} cm")


In [ ]:
B = 10_000
boot_rng = np.random.default_rng(123)
h1_values = df["h1"].to_numpy()
t_values = df["treatment"].to_numpy()

boot_ates = np.empty(B)
for b in range(B):
    idx = boot_rng.integers(0, N, N)
    boot_ates[b] = (
        h1_values[idx][t_values[idx] == 1].mean()
        - h1_values[idx][t_values[idx] == 0].mean()
    )

ci_95 = np.percentile(boot_ates, [2.5, 97.5])
print(f"IC 95% bootstrap = [{ci_95[0]:.3f}, {ci_95[1]:.3f}] cm")


In [ ]:
plt.hist(boot_ates, bins=40, color="#1e2bfa", alpha=0.75)
plt.axvline(ate_hat, color="#111111", lw=2, label="ATE_hat")
plt.axvline(ci_95[0], color="#dc2626", ls="--", label="IC 95%")
plt.axvline(ci_95[1], color="#dc2626", ls="--")
plt.xlabel("ATE (cm)")
plt.ylabel("frecuencia")
plt.legend(frameon=False)
plt.show()


El ATE es un número y el IC es frecuentista. Ninguno responde directamente:

- `P(ATE > 0 | datos)`
- `P(0.5 < ATE < 1.5 cm | datos)`
- `P(≥5 de 10 plantas nuevas > 14 cm | datos)`

Para eso pasamos a un modelo probabilístico y miramos la posterior.


## 3. Un modelo lineal en cm

Modelamos la altura final en centímetros:

$$
\begin{aligned}
h1_i &\sim \text{Normal}(\mu_i, \sigma) \\
\mu_i &= a + \beta_T T_i + \beta_H (h0_i - \overline{h0}) \\
a &\sim \text{Normal}(\overline{h1}, 5) \\
\beta_T &\sim \text{Normal}(0, 5) \\
\beta_H &\sim \text{Normal}(0, 1) \\
\sigma &\sim \text{HalfNormal}(5)
\end{aligned}
$$

`β_T` es el cambio promedio en altura final para una planta tratada, ajustando por altura inicial. Está en cm, así que es directamente comparable con el ATE de la materia. **No incluimos `F`**: es post-tratamiento y está en el camino causal.


In [ ]:
with pm.Model() as model_no_f:
    a = pm.Normal("a", mu=df["h1"].mean(), sigma=5)
    bT = pm.Normal("bT", mu=0, sigma=5)
    bH = pm.Normal("bH", mu=0, sigma=1)
    sigma = pm.HalfNormal("sigma", sigma=5)

    mu = (
        a
        + bT * df["treatment"].to_numpy()
        + bH * df["h0_c"].to_numpy()
    )
    pm.Normal("h1_obs", mu=mu, sigma=sigma, observed=df["h1"].to_numpy())

    trace_no_f = pm.sample(
        draws=2000,
        tune=2000,
        chains=4,
        target_accept=0.9,
        random_seed=RANDOM_SEED,
    )


In [ ]:
az.summary(
    trace_no_f,
    var_names=["a", "bT", "bH", "sigma"],
    hdi_prob=0.89,
).round(3)


In [ ]:
az.plot_trace(
    trace_no_f,
    var_names=["a", "bT", "bH", "sigma"],
    compact=True,
)
plt.tight_layout()
plt.show()


## 4. El modelo que controla por el hongo (mal)

`F` es post-tratamiento: está en el camino `T → F → H1`. Si lo incluimos, bloqueamos parte del efecto que queremos estimar. Este es el ejemplo de *post-treatment bias* del capítulo 6.


In [ ]:
with pm.Model() as model_with_f:
    a = pm.Normal("a", mu=df["h1"].mean(), sigma=5)
    bT = pm.Normal("bT", mu=0, sigma=5)
    bF = pm.Normal("bF", mu=0, sigma=5)
    bH = pm.Normal("bH", mu=0, sigma=1)
    sigma = pm.HalfNormal("sigma", sigma=5)

    mu = (
        a
        + bT * df["treatment"].to_numpy()
        + bF * df["fungus"].to_numpy()
        + bH * df["h0_c"].to_numpy()
    )
    pm.Normal("h1_obs", mu=mu, sigma=sigma, observed=df["h1"].to_numpy())

    trace_with_f = pm.sample(
        draws=2000,
        tune=2000,
        chains=4,
        target_accept=0.9,
        random_seed=RANDOM_SEED,
    )


In [ ]:
az.summary(
    trace_with_f,
    var_names=["a", "bT", "bF", "bH", "sigma"],
    hdi_prob=0.89,
).round(3)


In [ ]:
az.plot_forest(
    [trace_no_f, trace_with_f],
    model_names=["Sin F", "Con F"],
    var_names=["bT"],
    hdi_prob=0.89,
    figsize=(7, 3),
)
plt.show()


Sin `F`, `β_T` está cerca de 1.2 cm y el intervalo queda lejos de cero. Con `F`, `β_T` se acerca a cero y el intervalo incluye cero. El segundo modelo no es "más completo": es causalmente incorrecto para la pregunta.


## 5. Las tres preguntas causales

In [ ]:
bT_samples = trace_no_f.posterior["bT"].values.flatten()
hdi_89 = az.hdi(trace_no_f, var_names=["bT"], hdi_prob=0.89)["bT"].values

p_positive = (bT_samples > 0).mean()
p_practical = ((bT_samples > 0.5) & (bT_samples < 1.5)).mean()

print(f"β_T posterior = {bT_samples.mean():.3f} cm")
print(f"HDI 89% = [{hdi_89[0]:.3f}, {hdi_89[1]:.3f}] cm")
print(f"P(ATE > 0 | datos) = {p_positive:.3f}")
print(f"P(0.5 < ATE < 1.5 cm | datos) = {p_practical:.3f}")


In [ ]:
pd.DataFrame({
    "Pregunta": [
        "P(ATE > 0 | datos)",
        "P(0.5 < ATE < 1.5 cm | datos)",
    ],
    "Respuesta": [p_positive, p_practical],
}).round(3)


In [ ]:
az.plot_posterior(
    trace_no_f,
    var_names=["bT"],
    hdi_prob=0.89,
    ref_val=0,
    figsize=(8, 4),
)
plt.axvline(0.5, color="#dc2626", ls="--", label="0.5 cm")
plt.axvline(1.5, color="#dc2626", ls="--", label="1.5 cm")
plt.legend(frameon=False)
plt.show()


## 6. La pregunta predictiva: 10 plantas nuevas

Para cada draw de la posterior:

1. sampleamos la altura inicial de 10 plantas nuevas: `h0_new ~ Normal(10, 2)`
2. calculamos `mu_new = a + bT * 1 + bH * (h0_new - mean(h0))`
3. sampleamos `h1_new ~ Normal(mu_new, sigma)`
4. contamos cuántas superan `x = 14 cm`

La pregunta es `P(≥5 de 10 plantas tratadas > 14 cm | datos)`.


In [ ]:
N_NEW = 10
X_CM = 14.0

post = trace_no_f.posterior
a_s = post["a"].values.flatten()
bT_s = post["bT"].values.flatten()
bH_s = post["bH"].values.flatten()
sigma_s = post["sigma"].values.flatten()

pp_rng = np.random.default_rng(2026)
h0_new = pp_rng.normal(10, 2, size=(len(a_s), N_NEW))
h0_new_c = h0_new - df["h0"].mean()

mu_new = a_s[:, None] + bT_s[:, None] + bH_s[:, None] * h0_new_c
h1_new = pp_rng.normal(mu_new, sigma_s[:, None])
counts = (h1_new > X_CM).sum(axis=1)
p_at_least_5 = (counts >= 5).mean()

print(f"P(≥5 de 10 plantas tratadas > {X_CM:.0f} cm | datos) = {p_at_least_5:.3f}")


In [ ]:
plt.hist(
    counts,
    bins=np.arange(-0.5, 11.5, 1),
    color="#1e2bfa",
    alpha=0.75,
    rwidth=0.85,
)
plt.axvline(4.5, color="#dc2626", ls="--", label="5 o más")
plt.xlabel("plantas que superan 14 cm (de 10)")
plt.ylabel("frecuencia posterior")
plt.xticks(range(0, 11))
plt.legend(frameon=False)
plt.show()


In [ ]:
x_grid = np.arange(12, 17.5, 0.5)
p_grid = []

for x in x_grid:
    counts_x = (h1_new > x).sum(axis=1)
    p_grid.append((counts_x >= 5).mean())

plt.plot(x_grid, p_grid, marker="o", color="#1e2bfa")
plt.axvline(X_CM, color="#dc2626", ls="--", label="x = 14 cm")
plt.xlabel("umbral x (cm)")
plt.ylabel("P(≥5 de 10 > x | datos)")
plt.ylim(0, 1)
plt.legend(frameon=False)
plt.show()


In [ ]:
from scipy.stats import norm, binom

a_mean = a_s.mean()
bT_mean = bT_s.mean()
bH_mean = bH_s.mean()
sigma_mean = sigma_s.mean()

# Plug-in: fix parameters at posterior means and h0 at its mean.
mu_plugin = a_mean + bT_mean
p_single_plugin = 1 - norm.cdf(X_CM, loc=mu_plugin, scale=sigma_mean)
p_plugin = 1 - binom.cdf(4, N_NEW, p_single_plugin)

print(f"Plug-in: P(h1 > {X_CM:.0f} cm) = {p_single_plugin:.3f}")
print(f"Plug-in: P(≥5 de 10) = {p_plugin:.3f}")
print(f"Predictiva posterior: P(≥5 de 10) = {p_at_least_5:.3f}")


La predictiva posterior no es solo "el modelo con ruido": integra la incertidumbre en los parámetros y en la altura inicial de las plantas nuevas. Por eso puede diferir del plug-in que fija los parámetros en su media.


## 7. Chequeo predictivo posterior

In [ ]:
with model_no_f:
    ppc = pm.sample_posterior_predictive(trace_no_f, random_seed=RANDOM_SEED)

az.plot_ppc(ppc, num_pp_samples=100)
plt.show()


## 8. Conclusiones

- El ATE de la materia es un número: `ATE_hat = 1.05 cm`, IC 95% bootstrap `[0.13, 1.95]`.
- El modelo causal sin `F` devuelve `β_T = 1.20 cm [0.69, 1.71]` (89% HDI): el tratamiento hace crecer.
- El modelo con `F` devuelve `β_T = 0.03 cm [-0.28, 0.34]`: controlar una variable post-tratamiento borra el efecto.
- La posterior responde `P(ATE > 0) > 0.99` y `P(0.5 < ATE < 1.5) = 0.81`.
- La predictiva responde `P(≥5 de 10 plantas nuevas > 14 cm) = 0.78`.
- Bayes no elige el DAG: el DAG decide qué modelo es causal; la posterior cuantifica la incertidumbre.


In [ ]:
hdi_with_f = az.hdi(trace_with_f, var_names=["bT"], hdi_prob=0.89)["bT"].values
bT_with_f_samples = trace_with_f.posterior["bT"].values.flatten()

summary = pd.DataFrame({
    "Cantidad": [
        "ATE_hat (cm)",
        "IC 95% bootstrap",
        "β_T sin F (cm)",
        "β_T con F (cm)",
        "P(ATE > 0)",
        "P(0.5 < ATE < 1.5)",
        "P(≥5 de 10 > 14 cm)",
    ],
    "Valor": [
        f"{ate_hat:.2f}",
        f"[{ci_95[0]:.2f}, {ci_95[1]:.2f}]",
        f"{bT_samples.mean():.2f} [{hdi_89[0]:.2f}, {hdi_89[1]:.2f}]",
        f"{bT_with_f_samples.mean():.2f} [{hdi_with_f[0]:.2f}, {hdi_with_f[1]:.2f}]",
        f"{p_positive:.3f}",
        f"{p_practical:.3f}",
        f"{p_at_least_5:.3f}",
    ],
})
summary
